In [1]:

from dotenv import load_dotenv
load_dotenv()

True

In [ ]:

import os
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
os.environ['GOOGLE_API_KEY'] = os.getenv("GOOGLE_API_KEY")

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/Users/saicharan/Artificial Intellegence/artificialintellegence/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:

len(embeddings.embed_query("hello AI"))

384

In [5]:
from pinecone import Pinecone
pinecone_api_key = os.environ.get("PINECONE_API_KEY")

In [6]:

pinecone_api_key

'pcsk_3P6Hwg_Evve3sC997kXBe7hDJEJaF2N1S17kmAYPYp2kbtQYYGbRUAxi6kJok92vTLbfKc'

In [7]:
pc = Pinecone(api_key=pinecone_api_key)

In [14]:
from pinecone import ServerlessSpec

index_name = "my-index"  # change if desired

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [15]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding=embeddings)

In [16]:
from uuid import uuid4

from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
uuids = [str(uuid4()) for _ in range(len(documents))]
vector_store.add_documents(documents=documents, ids=uuids)

['3df334ae-5600-42bd-9923-83b7799ad5ef',
 '5f517bf4-685b-4808-93e7-e7da825d910f',
 '504b06ac-6950-422e-b200-839acd666a0d',
 '4ee039c6-29ae-4c3e-8d49-1f2ece8f72ab',
 '30163c96-11dc-46f6-aedc-6b5238f0a005',
 '68265040-8ad2-41ce-82e0-a45913c45bfc',
 '9cd3f885-2b49-4d79-a88c-233c4963fd90',
 '06301186-843a-4c8a-a692-495615e1e3ec',
 '0a949c77-fa02-49f0-8391-0510daca8884',
 'e2675411-4f82-4dce-a50f-daf9b4142d2c']

In [17]:
vector_store.delete(ids=[uuids[-1]])

In [18]:
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2,
    filter={"source": "tweet"},
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* Building an exciting new project with LangChain — check it out! [{'source': 'tweet'}]
* Building an exciting new project with LangChain — check it out! [{'source': 'tweet'}]


In [19]:
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 1, "score_threshold": 0.4},
)
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='79f1db81-962a-47fe-9634-3311c4250b79', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]